In [ ]:
import requests
import pandas as pd
from pandas import json_normalize
import math

# ✅ Set your token and upload ID here
token = input("Please set your NOMAD API token.")  
upload_id = "C4ynD5c9RXesSXOJf6oq_Q"

if not token or token == "your_nomad_token_here":
    print("❌ Please set your NOMAD API token above.")
else:
    base_url = "https://nomad-lab.eu/prod/v1/oasis/api/v1"
    
    # Step 1: Get metadata to determine total entries
    metadata_url = f"{base_url}/uploads/{upload_id}"
    metadata_response = requests.get(metadata_url, headers={'Authorization': f'Bearer {token}'})
    
    if metadata_response.status_code != 200:
        print(f"❌ Failed to fetch upload metadata: {metadata_response.status_code}")
    else:
        upload_metadata = metadata_response.json().get("data", {})
        total_entries = upload_metadata.get("entries", 0)
        page_size = 100
        max_offset = total_entries
        print(f"ℹ️ Found {total_entries} entries in upload {upload_id}")

        results = []

        # Step 2: Fetch paginated entry metadata
        for page_offset in range(0, max_offset, page_size):
            print(f"📦 Fetching entries at offset {page_offset}...")
            list_url = f"{base_url}/uploads/{upload_id}/entries?page_size={page_size}&page_offset={page_offset}"
            response = requests.get(list_url, headers={'Authorization': f'Bearer {token}'})
            
            if response.status_code != 200:
                print(f"⚠️ Failed to fetch page at offset {page_offset}: {response.status_code}")
                continue

            entries = response.json().get("data", [])

            for entry in entries:
                entry_id = entry.get("entry_id")
                detail_url = f"{base_url}/uploads/{upload_id}/entries/{entry_id}"
                detail_response = requests.get(detail_url, headers={'Authorization': f'Bearer {token}'})

                if detail_response.status_code != 200:
                    print(f"⚠️ Failed to fetch entry {entry_id}")
                    continue

                entry_data = detail_response.json().get("data", {})

                if entry_data.get("mainfile", "").endswith("classic.archive.json"):
                    continue

                try:
                    metadata = entry_data["entry_metadata"]["data"]
                    results.append(metadata)
                    print(f"✅ Added entry: {entry_id}")
                except Exception as e:
                    print(f"❌ Failed to extract metadata from {entry_id}: {e}")

        # Step 3: Save to CSV
        df = pd.DataFrame(json_normalize(results))
        output_file = f"bigrun.csv"
        df.to_csv(output_file, index=False)
        print(f"✅ Data saved to {output_file}")
